# 🚗 Car Market Trends Analysis — Notebook 1
## Data Cleaning & Inspection

**Project:** Car Market Trends Analysis with Car Dekho Data  
**Dataset:** 301 listings, 9 columns  
**Goal:** Load the dataset, inspect it thoroughly, flag quality issues, and produce a clean DataFrame ready for analysis.

---

### What this notebook covers
| Step | Task |
|------|------|
| 1 | Load the CSV file |
| 2 | Display shape, column names, and data types |
| 3 | Check for missing values |
| 4 | Check for duplicate rows |
| 5 | Validate the Year column |
| 6 | Validate Selling_Price and Present_Price |
| 7 | Validate Kms_Driven |
| 8 | Validate categorical columns |
| 9 | Apply all cleaning steps |
| 10 | Save the cleaned dataset |

---
## Step 0 — Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

---
## Step 1 — Load the CSV File

In [ ]:
# Build the path relative to this notebook's location
# The notebooks/ folder is one level below the project root
DATA_PATH = os.path.join('..', 'data', 'car_dekho_data.csv')

# Load raw data — we keep this as 'df_raw' and never modify it
df_raw = pd.read_csv(DATA_PATH)

print(f'File loaded: {os.path.abspath(DATA_PATH)}')
print(f'Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

---
## Step 2 — Structure: Shape, Columns, Data Types

In [ ]:
# Display first 5 rows
print('=== First 5 rows ===')
df_raw.head()

In [ ]:
# Column names and data types
print('=== Data Types ===')
print(df_raw.dtypes)
print()
print('=== Basic Statistics ===')
df_raw.describe()

**Observation:**
- `Car_Name`, `Fuel_Type`, `Seller_Type`, `Transmission` are `object` (text) columns
- `Year`, `Owner`, `Kms_Driven` are `int64` (integer numbers)
- `Selling_Price`, `Present_Price` are `float64` (decimal numbers)
- Prices are in **Indian Rupee Lakhs** (1 Lakh = Rs 1,00,000)

---
## Step 3 — Check for Missing Values

In [ ]:
# Count missing (NaN) values per column
missing = df_raw.isnull().sum()
pct     = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     pct
})
print(missing_df)

if missing.sum() == 0:
    print('\n✓ RESULT: No missing values found. Dataset is complete.')
else:
    print(f'\n⚠ RESULT: {missing.sum()} missing values found — action required.')

---
## Step 4 — Check for Duplicate Rows

In [ ]:
# Find rows where ALL column values are identical
n_dupes = df_raw.duplicated().sum()
print(f'Exact duplicate rows found: {n_dupes}')

if n_dupes > 0:
    print('\nDuplicate rows (all copies shown):')
    dupe_rows = df_raw[df_raw.duplicated(keep=False)]
    display(dupe_rows.sort_values(list(df_raw.columns)))
    print(f'\nDecision: These {n_dupes} duplicate(s) will be removed in Step 9.')

---
## Step 5 — Validate the Year Column

In [ ]:
print(f'Year range: {df_raw["Year"].min()} — {df_raw["Year"].max()}')
print(f'Unique years: {sorted(df_raw["Year"].unique())}')

# Flag anything before 1990 or after today
CURRENT_YEAR = 2024
invalid = df_raw[(df_raw['Year'] < 1990) | (df_raw['Year'] > CURRENT_YEAR)]

if len(invalid) > 0:
    print(f'\n⚠ {len(invalid)} rows with unusual Year:')
    print(invalid[['Car_Name', 'Year']])
else:
    print('\n✓ All Year values are valid (1990 – 2024).')

In [ ]:
# Visualise year distribution
fig, ax = plt.subplots(figsize=(12, 4))
yr_counts = df_raw['Year'].value_counts().sort_index()
ax.bar(yr_counts.index.astype(str), yr_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Number of Listings by Manufacture Year', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## Step 6 — Validate Selling_Price and Present_Price

In [ ]:
for col in ['Selling_Price', 'Present_Price']:
    print(f'{col}:')
    print(f'  Min    : Rs {df_raw[col].min():.2f}L')
    print(f'  Max    : Rs {df_raw[col].max():.2f}L')
    print(f'  Mean   : Rs {df_raw[col].mean():.2f}L')
    print(f'  Median : Rs {df_raw[col].median():.2f}L')

    bad = df_raw[df_raw[col] <= 0]
    if len(bad):
        print(f'  ⚠ {len(bad)} row(s) with price <= 0:')
        print(bad[['Car_Name', col]])
    else:
        print(f'  ✓ No zero or negative prices.')
    print()

# Check Selling_Price > Present_Price (unusual for used cars)
overpriced = df_raw[df_raw['Selling_Price'] > df_raw['Present_Price']]
print(f'Rows where Selling_Price > Present_Price: {len(overpriced)}')
if len(overpriced) == 0:
    print('✓ All selling prices are at or below present price.')

In [ ]:
# Side-by-side price distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_raw['Selling_Price'], bins=30, kde=True,
             color='steelblue', ax=axes[0])
axes[0].set_title('Selling Price Distribution')
axes[0].set_xlabel('Selling Price (Rs Lakhs)')

sns.histplot(df_raw['Present_Price'], bins=30, kde=True,
             color='darkorange', ax=axes[1])
axes[1].set_title('Present Price Distribution')
axes[1].set_xlabel('Present Price (Rs Lakhs)')

plt.tight_layout()
plt.show()
print('Both distributions are right-skewed (a few expensive cars pull the tail).')

---
## Step 7 — Validate Kms_Driven

In [ ]:
print(f'Kms_Driven — Min: {df_raw["Kms_Driven"].min():,}')
print(f'Kms_Driven — Max: {df_raw["Kms_Driven"].max():,}')
print(f'Kms_Driven — Mean: {df_raw["Kms_Driven"].mean():,.0f}')

# Flag potential outliers > 400,000 km
HIGH_KMS = 400_000
high = df_raw[df_raw['Kms_Driven'] > HIGH_KMS]
if len(high):
    print(f'\n⚠ {len(high)} row(s) with Kms_Driven > {HIGH_KMS:,}:')
    print(high[['Car_Name', 'Year', 'Kms_Driven']])
    print('\nDecision: Kept as-is. 500,000 km is plausible for a 16-year-old scooter.')

---
## Step 8 — Validate Categorical Columns

In [ ]:
# Expected valid values for each categorical column
valid_values = {
    'Fuel_Type'   : ['Petrol', 'Diesel', 'CNG'],
    'Seller_Type' : ['Dealer', 'Individual'],
    'Transmission': ['Manual', 'Automatic'],
}

for col, expected in valid_values.items():
    counts = df_raw[col].str.strip().value_counts()
    unique = df_raw[col].str.strip().unique().tolist()
    unexpected = [v for v in unique if v not in expected]

    print(f'--- {col} ---')
    print(counts.to_string())
    if unexpected:
        print(f'  ⚠ Unexpected values: {unexpected}')
    else:
        print(f'  ✓ All values are valid: {expected}')
    print()

# Owner distribution
print('--- Owner (number of previous owners) ---')
print(df_raw['Owner'].value_counts().sort_index())
print('  ✓ All Owner values are >= 0.' if df_raw['Owner'].min() >= 0 else '  ⚠ Negative Owner values found.')

In [ ]:
# Visualise categorical breakdown
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['Fuel_Type', 'Seller_Type', 'Transmission']):
    counts = df_raw[col].value_counts()
    ax.bar(counts.index, counts.values,
           color=['steelblue', 'darkorange', 'seagreen', 'crimson'][:len(counts)])
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha='center', fontsize=10)

plt.suptitle('Categorical Column Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 9 — Apply All Cleaning Steps

In [ ]:
# Always work on a COPY — never modify df_raw
df_clean = df_raw.copy()

# ── a) Strip whitespace from all text columns ─────────────────────────
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()
print('[a] Stripped whitespace from text columns.')

# ── b) Title-case Car_Name  ("swift" → "Swift") ──────────────────────
df_clean['Car_Name'] = df_clean['Car_Name'].str.title()
print('[b] Title-cased Car_Name.')

# ── c) Remove exact duplicate rows ───────────────────────────────────
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
removed = before - len(df_clean)
print(f'[c] Removed {removed} duplicate(s). Rows: {before} → {len(df_clean)}')

# ── d) Add Car_Age = 2024 - Year ─────────────────────────────────────
df_clean['Car_Age'] = 2024 - df_clean['Year']
print('[d] Added Car_Age column.')

# ── e) Add Price_Depreciation = Present_Price - Selling_Price ────────
df_clean['Price_Depreciation'] = (df_clean['Present_Price'] - df_clean['Selling_Price']).round(2)
print('[e] Added Price_Depreciation column.')

print(f'\nFinal cleaned shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')

---
## Step 10 — Before / After Summary

In [ ]:
print('=== CLEANING SUMMARY ===')
print(f'  Original rows    : {len(df_raw)}')
print(f'  Cleaned rows     : {len(df_clean)}')
print(f'  Rows removed     : {len(df_raw) - len(df_clean)}')
print(f'  Original columns : {df_raw.shape[1]}')
print(f'  Cleaned columns  : {df_clean.shape[1]}  (+{df_clean.shape[1]-df_raw.shape[1]} derived)')
print()
print('Final columns:')
for col in df_clean.columns:
    tag = ' ← derived' if col in ('Car_Age', 'Price_Depreciation') else ''
    print(f'  {col}{tag}')

In [ ]:
# Preview the cleaned DataFrame
df_clean.head(10)

In [ ]:
# Save cleaned data for use in other notebooks
CLEANED_PATH = os.path.join('..', 'data', 'car_dekho_cleaned.csv')
df_clean.to_csv(CLEANED_PATH, index=False)
print(f'Cleaned data saved to: {os.path.abspath(CLEANED_PATH)}')

---
## Summary of Findings

| Check | Result |
|-------|--------|
| Missing values | ✅ None |
| Duplicate rows | ⚠️ 2 found and removed (Ertiga 2016, Fortuner 2015) |
| Year validity | ✅ All values 2003–2018 |
| Price validity | ✅ No zero/negative, no selling > present |
| Kms_Driven outlier | ⚠️ 1 record at 500,000 km — kept (old scooter) |
| Categorical values | ✅ All match expected sets |
| **Final dataset** | **299 rows × 11 columns — ready for EDA** |